In [1]:
import torch
import gc

# Очищаем кэш GPU
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()


# Принудительный сборщик мусора
gc.collect()

# Проверяем результат
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"GPU memory cached: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

GPU memory allocated: 0.00 GB
GPU memory cached: 0.00 GB


In [2]:
import sys
import os

# Получаем путь к родительской директории проекта
project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
# Добавляем его в sys.path, если его там еще нет
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [3]:
import sys
import os
import torch
import numpy as np
from collections import defaultdict
from transformers import AutoModelForCausalLM, AutoTokenizer
import pandas as pd

import torch_pruning as tp

from fedcore.tools.index_resolver import wrap_parameters_with_resolver, IndexResolvingParameter

RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

BATCH_SIZE = 2
SEQ_LENGTH = 64
# MODEL_NAME = "google/gemma-2-2b"
MODEL_NAME = "Qwen/Qwen2.5-0.5B"

/home/user/projects/FedCore/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("Model loaded")

Model loaded


In [5]:
target_layer = model.model.layers[0]

In [6]:
wrap_parameters_with_resolver(
    target_layer,
    param_filter=lambda mod, name, param: name == 'weight' and isinstance(mod, torch.nn.Linear),
    aggregation_mode="union"
)
print("Parameters wrapped with IndexResolvingParameter")

# Проверка, что параметры действительно обёрнуты
for name, param in target_layer.named_parameters():
    if 'weight' in name and isinstance(param, IndexResolvingParameter):
        print(f"  {name}: original_to_current = {param.original_to_current}")

Parameters wrapped with IndexResolvingParameter
  self_attn.q_proj.weight: original_to_current = {0: None, 1: None, 2: None, 3: None, 4: None, 5: None, 6: None, 7: None, 8: None, 9: None, 10: None, 11: None, 12: None, 13: None, 14: None, 15: None, 16: None, 17: None, 18: None, 19: None, 20: None, 21: None, 22: None, 23: None, 24: None, 25: None, 26: None, 27: None, 28: None, 29: None, 30: None, 31: None, 32: None, 33: None, 34: None, 35: None, 36: None, 37: None, 38: None, 39: None, 40: None, 41: None, 42: None, 43: None, 44: None, 45: None, 46: None, 47: None, 48: None, 49: None, 50: None, 51: None, 52: None, 53: None, 54: None, 55: None, 56: None, 57: None, 58: None, 59: None, 60: None, 61: None, 62: None, 63: None, 64: None, 65: None, 66: None, 67: None, 68: None, 69: None, 70: None, 71: None, 72: None, 73: None, 74: None, 75: None, 76: None, 77: None, 78: None, 79: None, 80: None, 81: None, 82: None, 83: None, 84: None, 85: None, 86: None, 87: None, 88: None, 89: None, 90: None, 91

In [7]:
ignored_layers = []
for name, module in model.named_modules():
    if 'embed_tokens' in name or 'lm_head' in name:
        ignored_layers.append(module)
for i, layer in enumerate(model.model.layers):
    if layer != target_layer:
        ignored_layers.extend([layer.self_attn, layer.mlp, 
                               layer.input_layernorm, layer.post_attention_layernorm])

In [8]:
class ModelWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, input_ids, attention_mask, labels=None):
        return self.model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

wrapped_model = ModelWrapper(model).to(device)

In [9]:
example_inputs = {
    "input_ids": torch.randint(0, tokenizer.vocab_size, (BATCH_SIZE, SEQ_LENGTH)).to(device),
    "attention_mask": torch.ones(BATCH_SIZE, SEQ_LENGTH).to(device),
}
DG = tp.DependencyGraph().build_dependency(wrapped_model, example_inputs=example_inputs, ignored_layers=ignored_layers)
print("DependencyGraph built")


/home/user/projects/FedCore/.venv/lib/python3.10/site-packages/torch_pruning/dependency/graph.py:390: UserWarning: Unwrapped parameters detected: ['model.model.layers.2.input_layernorm.weight', 'model.model.layers.1.post_attention_layernorm.weight', 'model.model.layers.3.post_attention_layernorm.weight', 'model.model.layers.4.input_layernorm.weight', 'model.model.layers.7.post_attention_layernorm.weight', 'model.model.layers.9.post_attention_layernorm.weight', 'model.model.layers.10.input_layernorm.weight', 'model.model.norm.weight', 'model.model.layers.14.post_attention_layernorm.weight', 'model.model.layers.15.post_attention_layernorm.weight', 'model.model.layers.16.input_layernorm.weight', 'model.model.layers.20.post_attention_layernorm.weight', 'model.model.layers.21.post_attention_layernorm.weight', 'model.model.layers.22.input_layernorm.weight', 'model.model.layers.5.post_attention_layernorm.weight', 'model.model.layers.11.post_attention_layernorm.weight', 'model.model.layers.12.

DependencyGraph built


In [10]:
test_group = None
for group in DG.get_all_groups():
    for dep in group:
        mod = None
        if hasattr(dep.dep, 'source') and hasattr(dep.dep.source, 'module'):
            mod = dep.dep.source.module
        if mod is None and hasattr(dep.dep, 'target') and hasattr(dep.dep.target, 'module'):
            mod = dep.dep.target.module
        if mod is not None and hasattr(mod, 'weight'):
            if isinstance(mod.weight, IndexResolvingParameter):
                test_group = group
                break
    if test_group:
        break

if test_group is None:
    print("Group not found")
else:
    print("Found group with IndexResolvingParameter")


Found group with IndexResolvingParameter


In [11]:
pruner = tp.pruner.MetaPruner(
    model=wrapped_model,
    example_inputs=example_inputs,
    importance=tp.importance.MagnitudeImportance(p=2),
    pruning_ratio=0.5,
    ignored_layers=ignored_layers,
    round_to=8,
    iterative_steps=1
)

/home/user/projects/FedCore/.venv/lib/python3.10/site-packages/torch_pruning/dependency/graph.py:390: UserWarning: Unwrapped parameters detected: ['model.model.layers.2.input_layernorm.weight', 'model.model.layers.1.post_attention_layernorm.weight', 'model.model.layers.3.post_attention_layernorm.weight', 'model.model.layers.4.input_layernorm.weight', 'model.model.layers.7.post_attention_layernorm.weight', 'model.model.layers.9.post_attention_layernorm.weight', 'model.model.layers.10.input_layernorm.weight', 'model.model.norm.weight', 'model.model.layers.14.post_attention_layernorm.weight', 'model.model.layers.15.post_attention_layernorm.weight', 'model.model.layers.16.input_layernorm.weight', 'model.model.layers.20.post_attention_layernorm.weight', 'model.model.layers.21.post_attention_layernorm.weight', 'model.model.layers.22.input_layernorm.weight', 'model.model.layers.5.post_attention_layernorm.weight', 'model.model.layers.11.post_attention_layernorm.weight', 'model.model.layers.12.

In [12]:
imp = pruner.importance(test_group)   
print(f"Importance shape: {imp.shape}")

keep_ratio = 0.5
num_keep = int(len(imp) * keep_ratio)
_, important_indices = torch.topk(imp, num_keep)
important_indices = important_indices.tolist()
print(f"Keeping {num_keep} out of {len(imp)} channels")
print(f"Important indices (first 10): {important_indices[:10]}...")

mapping_updated = False
for dep in test_group:
    mod = None
    if hasattr(dep.dep, 'source') and hasattr(dep.dep.source, 'module'):
        mod = dep.dep.source.module
    if mod is None and hasattr(dep.dep, 'target') and hasattr(dep.dep.target, 'module'):
        mod = dep.dep.target.module
    if mod is None and hasattr(dep.dep, 'layer'):
        mod = dep.dep.layer
    if mod is None:
        continue

    if hasattr(mod, 'weight') and isinstance(mod.weight, IndexResolvingParameter):
        param = mod.weight
        param.resolve_indices(important_indices)
        print(f"Updated mapping for {param.module_name}")
        print(f"  Active groups: {sum(1 for v in param.original_to_current.values() if v is not None)} / {len(param.original_to_current)}")
        mapping_updated = True

if not mapping_updated:
    raise RuntimeError("No IndexResolvingParameter found in group.")


Importance shape: torch.Size([4864])
Keeping 2432 out of 4864 channels
Important indices (first 10): [1764, 596, 3565, 1774, 392, 1486, 1810, 2260, 173, 3222]...
Updated mapping for Linear.weight
  Active groups: 2432 / 4864
Updated mapping for Linear.weight
  Active groups: 2432 / 4864


In [13]:
print(param.current_to_original)

{0: 1, 1: 4, 2: 5, 3: 7, 4: 8, 5: 10, 6: 11, 7: 13, 8: 15, 9: 16, 10: 21, 11: 22, 12: 24, 13: 27, 14: 29, 15: 31, 16: 32, 17: 33, 18: 34, 19: 36, 20: 38, 21: 39, 22: 41, 23: 42, 24: 45, 25: 54, 26: 56, 27: 58, 28: 59, 29: 60, 30: 62, 31: 66, 32: 68, 33: 69, 34: 73, 35: 74, 36: 77, 37: 78, 38: 79, 39: 80, 40: 81, 41: 85, 42: 86, 43: 89, 44: 90, 45: 91, 46: 93, 47: 96, 48: 99, 49: 102, 50: 103, 51: 106, 52: 107, 53: 108, 54: 114, 55: 116, 56: 117, 57: 118, 58: 121, 59: 125, 60: 126, 61: 127, 62: 128, 63: 129, 64: 130, 65: 133, 66: 134, 67: 136, 68: 138, 69: 143, 70: 145, 71: 146, 72: 147, 73: 149, 74: 151, 75: 152, 76: 153, 77: 154, 78: 156, 79: 160, 80: 161, 81: 163, 82: 164, 83: 166, 84: 167, 85: 168, 86: 169, 87: 173, 88: 177, 89: 178, 90: 179, 91: 181, 92: 183, 93: 184, 94: 185, 95: 190, 96: 199, 97: 200, 98: 201, 99: 203, 100: 204, 101: 207, 102: 208, 103: 211, 104: 216, 105: 218, 106: 223, 107: 225, 108: 226, 109: 227, 110: 232, 111: 233, 112: 234, 113: 237, 114: 238, 115: 239, 116

In [13]:
import torch.nn.functional as F

In [15]:
for dep in test_group:
    mod = None
    if hasattr(dep.dep, 'source') and hasattr(dep.dep.source, 'module'):
        mod = dep.dep.source.module
    if mod is None and hasattr(dep.dep, 'target') and hasattr(dep.dep.target, 'module'):
        mod = dep.dep.target.module
    if mod is None and hasattr(dep.dep, 'layer'):
        mod = dep.dep.layer
    if mod is None:
        continue

    if hasattr(mod, 'weight') and isinstance(mod.weight, IndexResolvingParameter):
        param = mod.weight
        active = sum(1 for v in param.original_to_current.values() if v is not None)
        print(f"Active groups: {active} / {len(param.original_to_current)}")

        param.prune_out_channels()

        print(f"After prune: weight shape = {param.shape}")
        print(f"  is_contiguous: {param.is_contiguous()}")

        if param.dim() == 2:
            in_features = param.shape[1]
            x = torch.randn(2, in_features, device=param.device)
            bias = mod.bias if mod.bias is not None else None
            # 1. F.linear
            out = F.linear(x, param, bias)
            print(f"F.linear -> output shape {out.shape}, contiguous: {out.is_contiguous()}")
            # 2. F.relu
            relu_out = F.relu(out)
            print(f"F.relu -> output shape {relu_out.shape}")
            # 3. F.softmax (по последней оси)
            sm_out = F.softmax(out, dim=-1)
            print(f"F.softmax -> output shape {sm_out.shape}")
            # 4. F.layer_norm (нормализуем последнюю размерность out)
            ln_out = F.layer_norm(out, normalized_shape=(out.shape[-1],))
            print(f"F.layer_norm -> output shape {ln_out.shape}")
        else:
            print(f"Warning: weight dim={param.dim()} not tested with linear")

    if hasattr(mod, 'bias') and mod.bias is not None and isinstance(mod.bias, IndexResolvingParameter):
        param_bias = mod.bias
        param_bias.resolve_indices(important_indices)
        param_bias.prune()
        print(f"Bias after prune: shape {param_bias.shape}, contiguous: {param_bias.is_contiguous()}")


Active groups: 2432 / 4864


AttributeError: 'IndexResolvingParameter' object has no attribute 'prune_out_channels'

In [22]:
for dep in test_group:
    mod = None
    if hasattr(dep.dep, 'source') and hasattr(dep.dep.source, 'module'):
        mod = dep.dep.source.module
    if mod is None and hasattr(dep.dep, 'target') and hasattr(dep.dep.target, 'module'):
        mod = dep.dep.target.module
    if mod is None and hasattr(dep.dep, 'layer'):
        mod = dep.dep.layer
    if mod is None:
        continue

    if hasattr(mod, 'weight') and isinstance(mod.weight, IndexResolvingParameter):
        param = mod.weight
        param.resolve_indices(important_indices)
        print(f"Updated mapping for {param.module_name}")
        print(f"  Active groups: {sum(1 for v in param.original_to_current.values() if v is not None)} / {len(param.original_to_current)}")

for dep in test_group:
    mod = None
    if hasattr(dep.dep, 'source') and hasattr(dep.dep.source, 'module'):
        mod = dep.dep.source.module
    if mod is None and hasattr(dep.dep, 'target') and hasattr(dep.dep.target, 'module'):
        mod = dep.dep.target.module
    if mod is None and hasattr(dep.dep, 'layer'):
        mod = dep.dep.layer
    if mod is None:
        continue

    if not hasattr(mod, 'weight') or not isinstance(mod.weight, IndexResolvingParameter):
        continue

    param = mod.weight
    keep_original = list(param.current_to_original.values())   # исходные индексы активных групп
    if not keep_original:
        print(f"WARNING: keep_original empty for {param.module_name}, skipping")
        continue

    all_indices = set(range(param.num_original_groups))
    remove_idxs = list(all_indices - set(keep_original))
    print(f"\nPruning {param.module_name}: removing {len(remove_idxs)} channels")

    if isinstance(mod, torch.nn.Linear):
        pruned_module = tp.prune_linear_out_channels(mod, remove_idxs)
    elif isinstance(mod, torch.nn.Conv2d):
        pruned_module = tp.prune_conv_out_channels(mod, remove_idxs)
    else:
        print(f"  Unsupported module type: {type(mod)}")
        continue


    parent = None
    parent_attr = None
    for p_name, p_module in model.named_modules():
        for child_name, child in p_module.named_children():
            if child is mod:
                parent = p_module
                parent_attr = child_name
                break
        if parent is not None:
            break

    if parent is not None and parent_attr is not None:
        setattr(parent, parent_attr, pruned_module)
        print(f"  Replaced module in {parent_attr}")
    else:
        # Попробуем найти в target_layer напрямую
        for name, child in target_layer.named_children():
            if child is mod:
                setattr(target_layer, name, pruned_module)
                print(f"  Replaced module in target_layer.{name}")
                break
        else:
            print(f"  Could not find parent for {param.module_name}, module not replaced")
            continue

    new_weight = IndexResolvingParameter(
        data=pruned_module.weight.data,
        group_ids=keep_original,
        aggregation_mode=param.aggregation_mode,
        module_name=param.module_name
    )
    pruned_module.weight = new_weight
    print(f"  New weight shape: {pruned_module.weight.shape}, active groups: {len(keep_original)}")
    print(f"  is_contiguous: {pruned_module.weight.is_contiguous()}")

    if pruned_module.weight.dim() == 2:
        in_features = pruned_module.weight.shape[1]
        x = torch.randn(2, in_features, device=pruned_module.weight.device)
        bias = getattr(pruned_module, 'bias', None)
        out = F.linear(x, pruned_module.weight, bias)
        print(f"  F.linear output shape: {out.shape}")
        relu_out = F.relu(out)
        print(f"    F.relu -> output shape {relu_out.shape}")
            # 3. F.softmax (по последней оси)
        sm_out = F.softmax(out, dim=-1)
        print(f"    F.softmax -> output shape {sm_out.shape}")
            # 4. F.layer_norm (нормализуем последнюю размерность out)
        ln_out = F.layer_norm(out, normalized_shape=(out.shape[-1],))
        print(f"    F.layer_norm -> output shape {ln_out.shape}")

Updated mapping for Linear.weight
  Active groups: 2432 / 2432
Updated mapping for Linear.weight
  Active groups: 2432 / 2432

Pruning Linear.weight: removing 36 channels
  Replaced module in up_proj
  New weight shape: torch.Size([33, 896]), active groups: 2432
  is_contiguous: True
  F.linear output shape: torch.Size([2, 33])
    F.relu -> output shape torch.Size([2, 33])
    F.softmax -> output shape torch.Size([2, 33])
    F.layer_norm -> output shape torch.Size([2, 33])
